In [ ]:
cell = "K562"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
root_dir = "/mnt/d/TAD-MultiOmicsNet"

In [ ]:
import pandas as pd
import numpy as np
import json
import os

# Key: Load the normalization parameters of the GM12878 training set
GM12878_STATS = f"{root_dir}/GM12878/epi_signal/all/norm_stats.json"
with open(GM12878_STATS, "r") as f:
    stats = json.load(f)
print("使用 GM12878 训练集统计量做 Z-score：")
for k, v in stats.items():
    print(f"  {k}: mu={v['mu']:.4f}, sd={v['sd']:.4f}")

# # K562 raw epigenetic features tab file
TAB_PATH = f"{root_dir}/{cell}/epi_signal/epi4_K562_1kb.tab"

# Read and clean (column name processing identical to GM12878)
df = pd.read_csv(TAB_PATH, sep="\t")
df.columns = [col.replace("'", "").strip() for col in df.columns]
coord_key = "chr" if "chr" in df.columns else "#chr"
coord_cols = [coord_key, "start", "end"]
signal_cols = [c for c in df.columns if c not in coord_cols]
signal_cols_clean = [c.replace("K562","").strip("_") for c in signal_cols]
df.rename(columns=dict(zip(signal_cols, signal_cols_clean)), inplace=True)
signal_cols = signal_cols_clean

df[signal_cols] = df[signal_cols].apply(pd.to_numeric, errors="coerce")
df = df.sort_values(by=[coord_key, "start"], ascending=True).reset_index(drop=True)

df_log = df.copy()
df_log[signal_cols] = np.log1p(df_log[signal_cols]).clip(-3, 6)

# Key: Use the stats of GM12878 directly for Z-score calculation, do not recalculate.
df_norm = df_log.copy()
for c in signal_cols:
    mu = stats[c]["mu"]
    sd = stats[c]["sd"]
    mask_valid = ~df_norm[c].isna()
    if sd < 1e-8:
        df_norm.loc[mask_valid, c] = 0.0
    else:
        df_norm.loc[mask_valid, c] = (df_norm.loc[mask_valid, c] - mu) / sd

df_output = pd.concat([df[coord_cols], df_norm[signal_cols]], axis=1)

out_dir = f"{root_dir}/{cell}/epi_signal/"
test_chrs = [f"chr{i}" for i in range(1,23)] + ["chrX"]

def save_chrom_tsv(chr_name):
    save_path_dir = os.path.join(out_dir, "all")
    os.makedirs(save_path_dir, exist_ok=True)
    sub = df_output[df_output[coord_key]==chr_name].reset_index(drop=True)
    if sub.shape[0] == 0:
        print(f"{chr_name} No data, skip")
        return
    tsv_path = os.path.join(save_path_dir, f"{chr_name}_K562_epi1kb.tsv")
    sub.to_csv(tsv_path, sep="\t", index=False)
    print(f"Saved {chr_name} -> {tsv_path} | bin_count={sub.shape[0]}")

for chrom in test_chrs:
    save_chrom_tsv(chrom)

print("\n==== Preprocessing of the K562 cross-cell-line test set is completed ====")


In [ ]:
### K562 epigenetic features

In [ ]:
import pandas as pd
import numpy as np
import os

EPI_TSV = f"{root_dir}/{cell}/epi_signal/all/"
COORD_CSV = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/{cell}_coords_{display_reso}kb.csv"
OUT_DIR = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/Epi/"

signal_names = ["CTCF", "H3k4me3", "H3k27ac", "H3k27me3"]
n_seg_total = 100 
seg_length = 1000

os.makedirs(OUT_DIR, exist_ok=True)

hic_df = pd.read_csv(COORD_CSV)
chrs_order = [f"chr{i}" for i in range(1, 23)]
for chrom in chrs_order:
    group_df = hic_df[hic_df["chr"] == chrom]
    print(f"Processing {chrom}, total bins: {len(group_df)}")
    if len(group_df) == 0:
        print(f"!!! {chrom} no hic bins, skip")
        continue

    epi_tsv_path = os.path.join(EPI_TSV, f"{chrom}_{cell}_epi1kb.tsv")
    if not os.path.exists(epi_tsv_path):
        print(f"!!! {epi_tsv_path} not found, skip {chrom}")
        continue
    epi_df = pd.read_csv(epi_tsv_path, sep="\t")
    # 建立start -> 特征的字典，方便快速查询
    epi_dict = {}
    for _, row in epi_df.iterrows():
        s = int(row["start"])
        feats = row[signal_names].values.astype(np.float32)
        epi_dict[s] = feats

    chr_features = []
    chr_masks = []
    for _, bin_row in group_df.iterrows():
        bin_start = int(bin_row["start"])
        bin_end = int(bin_row["end"])
        center = bin_start + (bin_end - bin_start) / 2
        win_left = int(center - 50000)

        raw_100 = np.full((n_seg_total, len(signal_names)), np.nan)
        for seg_idx in range(n_seg_total):
            seg_l = win_left + seg_idx * seg_length
            epi_start_candidate = int(np.floor(seg_l / seg_length) * seg_length)
            if epi_start_candidate in epi_dict:
                raw_100[seg_idx] = epi_dict[epi_start_candidate]

        out_mask = (~np.isnan(raw_100).all(axis=1)).astype(np.float32)
        out_feat = np.nan_to_num(raw_100, nan=0.0).astype(np.float32)

        chr_features.append(out_feat)
        chr_masks.append(out_mask)

    feat_arr = np.stack(chr_features, axis=0)
    mask_arr = np.stack(chr_masks, axis=0)
    out_npy = os.path.join(OUT_DIR, f"epi{cell}_{chrom}_feat100.npy")
    out_mask = os.path.join(OUT_DIR, f"epi{cell}_{chrom}_mask100.npy")
    np.save(out_npy, feat_arr)
    np.save(out_mask, mask_arr)
    print(f"✅ {chrom} saved | feat shape:{feat_arr.shape}, mask shape:{mask_arr.shape}")

print("\n==== All chromosomes done ====")
